# Listing

> Directory listing components for list and grid views.

In [ ]:
#| default_exp components.listing

In [ ]:
#| export
from typing import Any, List, Optional, Set

from fasthtml.common import Div, Span, Table, Thead, Tbody, Tr, Th, P

# DaisyUI components
from cjm_fasthtml_daisyui.components.data_display.table import table, table_sizes
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui, text_dui, border_dui

# Tailwind utilities
from cjm_fasthtml_tailwind.utilities.spacing import p, m
from cjm_fasthtml_tailwind.utilities.sizing import w, h, min_h
from cjm_fasthtml_tailwind.utilities.typography import font_size, font_weight, text_align
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.borders import border, rounded
from cjm_fasthtml_tailwind.utilities.flexbox_and_grid import (
    flex_display, flex_direction, justify, items, gap, grow, grid_display, grid_cols
)
from cjm_fasthtml_tailwind.core.base import combine_classes

# Lucide icons
from cjm_fasthtml_lucide_icons.factory import lucide_icon

# Local imports
from cjm_file_discovery.core.models import FileInfo
from cjm_fasthtml_file_browser.core.models import DirectoryListing, BrowserState
from cjm_fasthtml_file_browser.core.config import (
    FileBrowserConfig, ViewMode, FileColumn, FilterConfig
)
from cjm_fasthtml_file_browser.components.item import (
    render_item, render_parent_item, BROWSER_ICONS
)

## Sorting and Filtering

Utilities for sorting and filtering file lists.

In [ ]:
#| export
def sort_files(
    files: List[FileInfo],           # Files to sort
    sort_by: str = "name",           # Sort field
    descending: bool = False,        # Sort direction
    folders_first: bool = True       # Sort folders before files
) -> List[FileInfo]:  # Sorted file list
    """Sort files by the specified field."""
    def sort_key(f: FileInfo):
        # Primary: folders first (if enabled)
        folder_order = 0 if f.is_directory else 1 if folders_first else 0
        
        # Secondary: sort field
        if sort_by == "name":
            value = f.name.lower()
        elif sort_by == "size":
            value = f.size or 0
        elif sort_by == "modified":
            value = f.modified or 0
        elif sort_by == "type":
            value = f.extension.lower() if f.extension else ""
        else:
            value = f.name.lower()
        
        return (folder_order, value)
    
    return sorted(files, key=sort_key, reverse=descending)


def filter_files(
    files: List[FileInfo],           # Files to filter
    filter_config: FilterConfig      # Filter configuration
) -> List[FileInfo]:  # Filtered file list
    """Filter files based on configuration."""
    return [f for f in files if filter_config.matches(f)]

In [ ]:
from cjm_file_discovery.core.models import FileType

# Test sorting
files = [
    FileInfo(name="zebra.txt", path="/zebra.txt", is_directory=False, size=100, extension="txt"),
    FileInfo(name="alpha", path="/alpha", is_directory=True),
    FileInfo(name="beta.py", path="/beta.py", is_directory=False, size=50, extension="py"),
]

# Sort by name, folders first
sorted_files = sort_files(files, sort_by="name", folders_first=True)
assert sorted_files[0].name == "alpha"  # Folder first
assert sorted_files[1].name == "beta.py"  # Then files alphabetically

# Sort by size
sorted_files = sort_files(files, sort_by="size", folders_first=False)
assert sorted_files[0].size == 0 or sorted_files[0].size is None  # Folder has no size

# Test filtering
filter_cfg = FilterConfig(allowed_extensions=[".py"])
filtered = filter_files(files, filter_cfg)
assert len(filtered) == 2  # One .py file + folder (folders pass extension filter)
assert any(f.name == "beta.py" for f in filtered)
assert any(f.name == "alpha" for f in filtered)

print("Sorting and filtering tests passed!")

Sorting and filtering tests passed!


## Empty State

Component shown when a directory is empty or has no matching files.

In [ ]:
#| export
def render_empty_state(
    message: str = "No files found",  # Message to display
    icon_name: str = "folder-open"    # Icon to show
) -> Any:  # Empty state component
    """Render empty state for when directory has no items."""
    return Div(
        Div(
            lucide_icon(icon_name, size=12, cls=str(text_dui.base_content.opacity(30))),
            cls=str(m.b(4))
        ),
        P(
            message,
            cls=combine_classes(text_dui.base_content.opacity(50), font_size.lg)
        ),
        cls=combine_classes(
            flex_display, flex_direction.col, items.center, justify.center,
            p(8), min_h(48)
        )
    )

In [ ]:
from fasthtml.common import to_xml

# Test empty state
empty = render_empty_state("This folder is empty")
html = to_xml(empty)
assert "This folder is empty" in html
assert "<svg" in html.lower()

print("Empty state tests passed!")

Empty state tests passed!


## Error State

Component shown when there's an error accessing a directory.

In [ ]:
#| export
def render_error_state(
    error_message: str  # Error message to display
) -> Any:  # Error state component
    """Render error state for when directory access fails."""
    return Div(
        Div(
            lucide_icon("circle-alert", size=12, cls=str(text_dui.error)),
            cls=str(m.b(4))
        ),
        P(
            error_message,
            cls=combine_classes(text_dui.error, font_size.lg)
        ),
        cls=combine_classes(
            flex_display, flex_direction.col, items.center, justify.center,
            p(8), min_h(48)
        )
    )

In [ ]:
# Test error state
error = render_error_state("Permission denied")
html = to_xml(error)
assert "Permission denied" in html
assert "text-error" in html

print("Error state tests passed!")

Error state tests passed!


## List View Table Header

Header row for the list view table.

In [ ]:
#| export
def _render_list_header(
    config: FileBrowserConfig,             # Browser configuration
    has_selectable_files: bool = False     # Whether any files can be selected
) -> Any:  # Table header component
    """Render the header row for list view."""
    headers = []
    
    # Checkbox column (if selection enabled)
    if config.selection_mode.value != "none" and has_selectable_files:
        headers.append(Th("", cls=str(w(8))))
    
    # Column headers based on config
    column_labels = {
        FileColumn.NAME: "Name",
        FileColumn.SIZE: "Size",
        FileColumn.MODIFIED: "Modified",
        FileColumn.TYPE: "Type",
        FileColumn.EXTENSION: "Extension",
        FileColumn.PATH: "Path",
    }
    
    # Name is always first
    headers.append(Th("Name"))
    
    # Other columns
    for col in config.view.columns:
        if col != FileColumn.NAME:  # Name already added
            headers.append(Th(
                column_labels.get(col, col.value.title()),
                cls=combine_classes(text_nowrap) if col in [FileColumn.SIZE, FileColumn.MODIFIED] else None
            ))
    
    return Thead(Tr(*headers))

# Import for the header
from cjm_fasthtml_tailwind.utilities.typography import text_nowrap

In [ ]:
# Test list header
config = FileBrowserConfig()
header = _render_list_header(config, has_selectable_files=True)
html = to_xml(header)
assert "Name" in html
assert "Size" in html
assert "Modified" in html

print("List header tests passed!")

List header tests passed!


## List View

Renders directory contents as a table.

In [ ]:
#| export
def render_list_view(
    listing: DirectoryListing,              # Directory listing to render
    config: FileBrowserConfig,              # Browser configuration
    state: BrowserState,                    # Current browser state
    navigate_url: Optional[str] = None,     # URL for directory navigation
    select_url: Optional[str] = None,       # URL for file selection
    hx_target: Optional[str] = None,        # HTMX target for navigate swaps
    listing_id: Optional[str] = None,       # HTML ID for the listing container
    select_hx_target: Optional[str] = None, # HTMX target for select swaps (preserves scroll)
    select_hx_swap: Optional[str] = None,   # HTMX swap mode for select (e.g. "innerHTML")
) -> Any:  # List view component
    """Render directory contents as a table/list view."""
    # Handle error state
    if listing.error:
        return render_error_state(listing.error)
    
    # Filter and sort files
    files = filter_files(listing.items, config.filter)
    files = sort_files(
        files,
        sort_by=state.sort_by,
        descending=state.sort_descending,
        folders_first=config.view.sort_folders_first
    )
    
    # Check if any files can be selected
    has_selectable = any(config.can_select(f) for f in files)
    
    # Build rows
    rows = []
    
    # Parent navigation row
    if listing.parent_path and config.show_parent_navigation:
        rows.append(render_parent_item(
            listing.parent_path,
            ViewMode.LIST,
            navigate_url or "",
            hx_target
        ))
    
    # File/folder rows
    from cjm_fasthtml_file_browser.core.html_ids import FileBrowserHtmlIds
    for idx, file_info in enumerate(files):
        is_selected = state.selection.is_selected(file_info.path)
        rows.append(render_item(
            file_info,
            config,
            ViewMode.LIST,
            is_selected=is_selected,
            item_id=FileBrowserHtmlIds.item_id(idx),
            navigate_url=navigate_url,
            select_url=select_url,
            hx_target=hx_target,
            select_hx_target=select_hx_target,
            select_hx_swap=select_hx_swap,
        ))
    
    # Empty state
    if not rows:
        return render_empty_state()
    
    # Build table
    table_attrs = {
        "cls": combine_classes(table, table_sizes.sm, w.full)
    }
    if listing_id:
        table_attrs["id"] = listing_id
    
    return Table(
        _render_list_header(config, has_selectable),
        Tbody(*rows),
        **table_attrs
    )

In [ ]:
# Test list view
from cjm_fasthtml_file_browser.core.models import BrowserSelection

config = FileBrowserConfig()
state = BrowserState(current_path="/home/user")

# Create a listing
listing = DirectoryListing(
    path="/home/user",
    items=[
        FileInfo(name="folder", path="/home/user/folder", is_directory=True),
        FileInfo(name="file.txt", path="/home/user/file.txt", is_directory=False, size=1024),
    ],
    parent_path="/home"
)

# Render list view
list_view = render_list_view(listing, config, state, navigate_url="/nav", select_url="/sel")
html = to_xml(list_view)
assert "<table" in html.lower()
assert "folder" in html
assert "file.txt" in html
assert ".." in html  # Parent navigation

# Test error state
error_listing = DirectoryListing(path="/restricted", items=[], error="Permission denied")
error_view = render_list_view(error_listing, config, state)
html = to_xml(error_view)
assert "Permission denied" in html

# Test empty state
empty_listing = DirectoryListing(path="/empty", items=[])
empty_view = render_list_view(empty_listing, config, state)
html = to_xml(empty_view)
assert "No files found" in html

print("List view tests passed!")

List view tests passed!


## Grid View

Renders directory contents as a grid of cards.

In [ ]:
#| export
def render_grid_view(
    listing: DirectoryListing,              # Directory listing to render
    config: FileBrowserConfig,              # Browser configuration
    state: BrowserState,                    # Current browser state
    navigate_url: Optional[str] = None,     # URL for directory navigation
    select_url: Optional[str] = None,       # URL for file selection
    hx_target: Optional[str] = None,        # HTMX target for navigate swaps
    listing_id: Optional[str] = None,       # HTML ID for the listing container
    select_hx_target: Optional[str] = None, # HTMX target for select swaps (preserves scroll)
    select_hx_swap: Optional[str] = None,   # HTMX swap mode for select (e.g. "innerHTML")
) -> Any:  # Grid view component
    """Render directory contents as a grid of cards."""
    # Handle error state
    if listing.error:
        return render_error_state(listing.error)
    
    # Filter and sort files
    files = filter_files(listing.items, config.filter)
    files = sort_files(
        files,
        sort_by=state.sort_by,
        descending=state.sort_descending,
        folders_first=config.view.sort_folders_first
    )
    
    # Build cards
    cards = []
    
    # Parent navigation card
    if listing.parent_path and config.show_parent_navigation:
        cards.append(render_parent_item(
            listing.parent_path,
            ViewMode.GRID,
            navigate_url or "",
            hx_target
        ))
    
    # File/folder cards
    from cjm_fasthtml_file_browser.core.html_ids import FileBrowserHtmlIds
    for idx, file_info in enumerate(files):
        is_selected = state.selection.is_selected(file_info.path)
        cards.append(render_item(
            file_info,
            config,
            ViewMode.GRID,
            is_selected=is_selected,
            item_id=FileBrowserHtmlIds.item_id(idx),
            navigate_url=navigate_url,
            select_url=select_url,
            hx_target=hx_target,
            select_hx_target=select_hx_target,
            select_hx_swap=select_hx_swap,
        ))
    
    # Empty state
    if not cards:
        return render_empty_state()
    
    # Build grid
    num_cols = config.view.grid_columns
    grid_cls = combine_classes(
        grid_display,
        f"grid-cols-{num_cols}",  # Dynamic grid columns
        gap(4),
        p(4)
    )
    
    grid_attrs = {"cls": grid_cls}
    if listing_id:
        grid_attrs["id"] = listing_id
    
    return Div(*cards, **grid_attrs)

In [ ]:
# Test grid view
config = FileBrowserConfig()
state = BrowserState(current_path="/home/user")

listing = DirectoryListing(
    path="/home/user",
    items=[
        FileInfo(name="folder", path="/home/user/folder", is_directory=True),
        FileInfo(name="image.png", path="/home/user/image.png", is_directory=False, size=2048),
    ],
    parent_path="/home"
)

# Render grid view
grid_view = render_grid_view(listing, config, state, navigate_url="/nav")
html = to_xml(grid_view)
assert "grid" in html  # Grid layout
assert "folder" in html
assert "image.png" in html
assert ".." in html  # Parent navigation

print("Grid view tests passed!")

Grid view tests passed!


## Unified Listing Render

Single function that dispatches to list or grid view based on view mode.

In [ ]:
#| export
def render_listing(
    listing: DirectoryListing,              # Directory listing to render
    config: FileBrowserConfig,              # Browser configuration
    state: BrowserState,                    # Current browser state
    navigate_url: Optional[str] = None,     # URL for directory navigation
    select_url: Optional[str] = None,       # URL for file selection
    hx_target: Optional[str] = None,        # HTMX target for navigate swaps
    listing_id: Optional[str] = None,       # HTML ID for the listing container
    select_hx_target: Optional[str] = None, # HTMX target for select swaps (preserves scroll)
    select_hx_swap: Optional[str] = None,   # HTMX swap mode for select (e.g. "innerHTML")
) -> Any:  # Listing component (table or grid)
    """Render directory listing based on current view mode."""
    view_mode = ViewMode(state.view_mode)
    
    if view_mode == ViewMode.LIST:
        return render_list_view(
            listing, config, state,
            navigate_url, select_url, hx_target, listing_id,
            select_hx_target, select_hx_swap,
        )
    else:
        return render_grid_view(
            listing, config, state,
            navigate_url, select_url, hx_target, listing_id,
            select_hx_target, select_hx_swap,
        )

In [ ]:
# Test unified render
config = FileBrowserConfig()
listing = DirectoryListing(
    path="/home/user",
    items=[FileInfo(name="test.txt", path="/test.txt", is_directory=False)]
)

# List mode
state_list = BrowserState(current_path="/home/user", view_mode="list")
result = render_listing(listing, config, state_list)
assert "<table" in to_xml(result).lower()

# Grid mode
state_grid = BrowserState(current_path="/home/user", view_mode="grid")
result = render_listing(listing, config, state_grid)
assert "grid" in to_xml(result)

print("Unified render tests passed!")

Unified render tests passed!


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()